# Descarga de Datos

Como primera etapa del analisis exploratorio de datos. Vamos a descargar la información de la Direccion General del Servicio de Información Agroalimentaria y Pesquera. Para ello realizaremos el acceso mediante 'dgsiap.py'

In [1]:
from scripts import dgsiap
from datetime import date
import yarl
import re
import requests

Bienvenido al script de descarga de datos abiertos del DGSIAP.
Conexion Establecida


Una vez establecida la conexión. Seguiremos con la selección de datos.

## Introducción

Primero crearemos un ligero mensaje para introducir la fuente de información.

In [2]:
# Definimos nuestra zona de trabajo.
content = dgsiap.page.find("main")
# Dividiremos la pagina en sus secciones principales
sections = content.find_all("section", recursive=False)

En este caso la pagina web nos permitio asociar cada bloque "section" con una bloque importante de la pagina del DGSIAP

* Datos generales de la fuente.
* Introducción breve
* Ultima publicación. (El conjunto de datos más reciente)
* Información por nivel. (Datos históricos)

En esta sección trabajaremos las primeras 2 categorias. El resto se verán en la descarga de los datos.

### Datos Generales

Explorando la pagina web podemos obtener información relevante para el analisis.

In [3]:
tree = sections[0]

intro_data = []
for t in tree.find_all("li"):
    if t.text!="":
        intro_data.append(t.text)

print(f"De parte de la {intro_data[0]} te damos la bienvenida al conjunto de {intro_data[1]} de la {intro_data[2]}")

De parte de la Dirección General del Servicio de Información Agroalimentaria y Pesquera te damos la bienvenida al conjunto de Datos abiertos de la Información agrícola


Como podemos ver en esta parte. El conjunto de datos forma parte de los datos Abiertos de la DGSIAP. Una dependencia del gobierno de México.

### Breve introducción.

El DGSIAP ya nos da un pequeño adelante de lo veremos en los datos. 

In [4]:
intro = sections[1]
raw_intro=list()
for t in intro.find_all("div", recursive=False):
    if t.text!="":
        raw_block=(t.text).split("\t")
        for raw in raw_block:
            if raw.strip() != "":
                raw_intro.append(raw)

print(f"Conjuntos de {raw_intro[0]} sobre {raw_intro[1]}")
print(raw_intro[-1])

Conjuntos de Datos abiertos sobre Estadística de Producción Agrícola

Los usuarios de la información podrán disponer de los datos anualizados de estadística básica agrícola a nivel nacional, estatal, por Distrito de Desarrollo Rural y municipal, la cual será de gran utilidad para el análisis del subsector y la toma de decisiones públicas y privadas que contribuyan al desarrollo rural sustentable del país.



Lo anterior solo es la introducción hacia el conjunto de datos. Ahora profundizaremos en los datos mediante los ultimos bloques.

* Última publicación.
* Información por nivel.

## Datos
Dentro del conjunto de datos se encontraran tablas en función de un cierto parámetro. El DGSIAP decidio clasificarlos en función del tipo.
* Información Principal. (Municipal y Nacional en la página web).
* No seguimiento. (Registro de elementos agricolas no prioritarios).

En el caso de la información principal está se divide por región
* Nivel Municipal.
* Nivel Nacional.
mientras la sección de no seguimiento solo posee a nivel nacional. De forma indistinta, cada una posee una subdivisión anual.

Cada bloque antes mencionado
* Nivel Municional.
* Nivel Nacional.
* No seguimiento 
posee un diccionario de datos que nos ayudara a interpretar mejor la información más adelante.

### Ultima publicación.
Como su nombre indica, corresponde a los datos más recientes. Para cada tabla de datos a descargar y usar debemos obtener su respectivo diccionario de datos.

In [5]:
last = sections[2]
search_title_section=last.find("div")
last_title = search_title_section.text.strip("\n").strip("\t")[:-1]
last_title

'Última publicación'

#### Diccionario de Datos.

Para conocer mejor la base de datos a trabajar comenzaremos obteniendo el diccionario de datos de la ultima publicación.

In [35]:
def get_catalog(sec=sections[2]):
    block = sec.find("div", {"class":"dict-section"})
    mini_blocks = block.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
    codebook, metadata = mini_blocks[0], mini_blocks[1]
    codebook_info = {"title": codebook.find("h2").text, 
                     "description": codebook.find("p").text,
                     "download": codebook.find("a")['href']}
    meta = metadata.find("div")
    datetype = meta.find_all("div", recursive=False)
    metadata_info = {"title": meta.find("h4").text,
                     "state": datetype[0].find("span", {"class":"label"}).text,
                     "lastdate": datetype[0].find("span", {"class":"value"}).text,
                     "range": datetype[1].find("span", {"class":"label"}).text,
                     "size": datetype[1].find("span", {"class":"value"}).text
                    }
    return metadata_info, codebook_info

last_metadata, last_codebook = get_catalog()
print(f"{last_title}")
print(f"{last_codebook['title']}")
print(f"{last_metadata['state']} el dia {last_metadata['lastdate']}")
print(f"{last_codebook['description']}")
print(f"Link de Descarga {last_codebook['download']}")

Última publicación
Diccionario de datos
Actualizado el dia 12 agosto 2026
Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.
Link de Descarga /index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7


#### Tablas de Datos.

La descarga de los diccionarios lo revisaremos durante esta sección para conglomerar todas las descargas. Asi que procedemos con la descarga de los archivos.

In [7]:
def get_data(sec=sections[2]):
    block = sec.find("div", {"class":"bento-card h-100"})
    u = block.find_all("div", recursive=False)
    data_info = [s.text for s in u[0].find_all("span", recursive=False)]
    data_get = u[1]
    data_link = data_get.find("a")['href']
    return data_info, data_link

get_data()

(['Nivel Municipal', 'RECIENTE 2025'],
 '/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025')

Como se ve, obtenemos un poco más de información de los datos. Siendo que la versión más reciente de Producción Agricola dada por el DGSIAP fue en el 2025 a nivel municipal. Notemos como en este caso y en el diccionario de datos. Las funciones ´get_data()' y ´get_catalog()´ devuelven links incompletos. Esto fue hecho a proposito para permitir al usuario consultar la información de interés antes de descargarla. Despues de revisar los datos historicos procederemos con la descarga.

### Datos Historicos. Información por Nivel.

Como observamos en la pagina. Y en esta libreta. Los datos historicos se dividen en 3 bloques.

In [46]:
def blocks_intro():
    history = sections[3]
    data_title = history.find("div", {"class":"mb-0 h5 font-weight-bold section-title"}).text
    data_content = history.find("div", {"id":"dataAccordion"})
    blocks = data_content.find_all("div", recursive=False)
    data_blocks = {b.find("div", {"class":"d-flex align-items-center"}).text.strip(): b for n, b in enumerate(blocks)}
    print(f"Los datos historicos se presentan en el espacio denomiado: {data_title}")
    data_blocks_f = [f"{k + 1}. {b}" for k, b in enumerate(data_blocks.keys())]
    print(f"Los datos se encuentran divididos en 3 bloques:\n\n{"\n".join(data_blocks_f)}")
    return data_blocks

data_blocks =blocks_intro()

Los datos historicos se presentan en el espacio denomiado: Información por nivel
Los datos se encuentran divididos en 3 bloques:

1. Municipal
2. Nacional
3. No seguimiento


Entonces nos interesa poder elegir el bloque a trabajar. De igual forma que en la ultima publicación debemos extraer los diccionarios respectivos. Notemos que al ser la más reciente la de Nivel Municipal. El bloque Municipal no requiere extracción adicional. Entonces extraremos la de Nacional y de No seguimiento.

### Diccionario de Datos.

Aqui es **IMPORTANTE** señalar la "fortuna" de la buena estructura de la página del DGSIAP. La obtención no diferira mucho de lo visto previamente.

In [47]:
data_dicts = {db:[] for db in data_blocks.keys()}

for db in list(data_dicts.keys())[1:]:
    block = data_blocks[db]
    dblock = block.find("div", {"class":"dict-section"})
    datadict = dblock.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
    break

Si recordamos la función definida previamente

```python
def get_catalog(sec=sections[2]):
    block = sec.find("div", {"class":"dict-section"})
    mini_blocks = block.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
    codebook, metadata = mini_blocks[0], mini_blocks[1]
    codebook_info = {"title": codebook.find("h2").text, 
                     "description": codebook.find("p").text,
                     "download": codebook.find("a")['href']}
    meta = metadata.find("div")
    datetype = meta.find_all("div", recursive=False)
    metadata_info = {"title": meta.find("h4").text,
                     "state": datetype[0].find("span", {"class":"label"}).text,
                     "lastdate": datetype[0].find("span", {"class":"value"}).text,
                     "range": datetype[1].find("span", {"class":"label"}).text,
                     "size": datetype[1].find("span", {"class":"value"}).text
                    }
    return codebook_info, metadata_info
```

Vemos que es igual a nuestro comienzo. Entonces, podemos obtener los diccionarios sin nuevo código


In [48]:
# "metadata":None ,"info":"None"

data_dicts = {db:{} for db in data_blocks.keys()}

history_blocknames = list(data_dicts.keys())
for db in history_blocknames:
    if db == history_blocknames[0]:
        data_dicts[db]["metadata"], data_dicts[db]["metadata"] = get_catalog()
        continue
    block = data_blocks[db]
    data_dicts[db]["metadata"], data_dicts[db]["info"] =  get_catalog(sec=block)

data_dicts


{'Municipal': {'metadata': {'title': 'Diccionario de datos',
   'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.',
   'download': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7'}},
 'Nacional': {'metadata': {'title': 'Metadatos Principales',
   'state': 'Actualizado',
   'lastdate': '29 julio 2025',
   'range': 'Cobertura 1980 - 2002',
   'size': 'Nivel Nacional'},
  'info': {'title': 'Diccionario de datos',
   'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.',
   'download': '/index.php?view=3494D63A-569A6D70-7435040E-4965D1CB'}},
 'No seguimiento': {'metadata': {'title': 'Metadatos Principales',
   'state': 'Actualizado',
   'lastdate': '29 julio 2025',
   'range': 'Cobertura 2020 - 2021',
   'size': 'Nivel Nacional'},
  'info': {'title': 'Diccionario de datos',
   'description': 'Consulte cada nombre de 

### Conjunto de datos.

Para la descarga de datos historicos nos conviene aprovechar su estrucutra de bloques antes que intentar reutilizar código previo.

In [51]:

for db in data_history.keys():
    block = data_blocks[db]
    con_data = block.find_all("a", {"class":"dl-btn"})
    href_data = [yarl.URL(a['href']) for a in con_data]
    data_history[db] += href_data

print(data_history)

{'Municipal': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2024'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2023'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2022'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2021'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2018'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2017'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2016'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2015'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2014'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2013'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2012'), URL('

## Descarga de Datos

A lo largo de esta libreta vimos como obtener los url de cada elemento de la pagina del DGSIAP.

In [58]:
# Conglomerado en una solo dicctionario. 
assert set(data_history.keys()) == set(data_dicts.keys())

# print(data_history)
#print(data_dicts)
dgsiap_data = {b: {"Metadata": data_dicts[b], "Datasets": data_history[b]} for b in data_dicts.keys()} 

print(dgsiap_data['Municipal'])
print(dgsiap_data['Nacional']['Datasets'])


{'Metadata': {'metadata': {'title': 'Diccionario de datos', 'description': 'Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.', 'download': '/index.php?view=5D24F995-E734D711-1EA6F8D2-9D1387D7'}}, 'Datasets': [URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2025'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2024'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2023'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2022'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2021'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2020'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2019'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2018'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2017'), URL('/index.php?view=10AE434F-A2158368-A120BC5A-EDF4AFAA&ANIO=2016'), URL('/index.php?vie

In [ ]:
def parse_link(url, base=dgsiap.DOWNLOAD_URL):
    y_url= yarl.URL(url)
    link = base.with_query(y_url.query)
    return link

def get_filename(link):
    head = requests.head(link)
    for u, v in head.headers.items():
        try:
            if v.index("filename"):
                name=v.split("filename=")[-1]
                name=name[1:-1]
                break
        except:
            pass
    return name
    


def download(link, filename, path=dgsiap.DATA_PATH, **kwargs):
    hint = kwargs.get("hint", 0)
    filelink = link / filename
    try:
        odata = requests.get(link)
    except e:
        print(f"Error {e}. Status Code {odata.status_code}")
    filepath = path / filename
    if hint:
        check = input(f"El archivo {filename} se guardara en {filepath}.\n¿Deseas continuar? Responda NO si quiere cancelar la descarga")
        if check == "NO":
            print("Revise el archivo a descargar\n y/o la ruta donde guardarlos en dgsiap.py")
    print("La conexión procedio. Comenzamos con la descarga")
    print(filepath)
    with odata as data:
        with open(filepath, "wb") as f:
            f.write(data.content)
    print(f"Archivo {filename} descargado correctamente")
    print(f"Descargado el {date.today()}")
    print(f"Se encuentra en {filepath}")


info, url = get_data()
link = parse_link(url)
name = get_filename(link)
download(link, name)


La conexión procedio. Comenzamos con la descarga
/home/david/Documents/Repositories/academica/proyecto_agronomia/scripts/../data/external/Cierre_agricola_mun_2025.csv
Archivo Cierre_agricola_mun_2025.csv descargado correctamente
Descargado el 2026-09-23
Se encuentra en /home/david/Documents/Repositories/academica/proyecto_agronomia/scripts/../data/external/Cierre_agricola_mun_2025.csv
